In [ ]:
!pip install --upgrade pip setuptools wheel
!pip install seqeval

  Using cached seqeval-1.2.2.tar.gz (43 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16251 sha256=8e2be07a5042e32d0af61f8516936fb5e6af3d560fd4708260d81bf853cd25fd
  Stored in directory: /root/.cache/pip/wheels/14/cf/a7/8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully built seqeval


In [ ]:
# ============================================================
# STAGE 3: MULTI-TASK DICTABERT-LARGE
# ============================================================
import json
import os
import shutil
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from google.colab import drive
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModel,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

# ------------------------------------------------------------------------------
# 1. SETUP DIRECTORIES & MODEL CONFIG
# ------------------------------------------------------------------------------
print("💾 Mounting Google Drive...")
drive.mount("/content/drive")

MODEL_NAME = "dicta-il/dictabert-large"
cols = ["dim", "tr", "fr", "sr", "fe", "refLevel", "ss"]

BASE_MODEL_DIR = "/content/drive/MyDrive/Workshop/New_Models/stage3_dictabert_uncertainty"
LARGE_MODEL_DIR = "/content/drive/MyDrive/Workshop/Large/models/stage3_dictabert_large_uncertainty"
TMP_CHECKPOINT_DIR = "/tmp/stage3_large_checkpoints"

os.makedirs(LARGE_MODEL_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------------------------
# 2. ENCODERS & CLASS WEIGHTS
# ------------------------------------------------------------------------------
train_path = "train_set.csv" if os.path.exists("train_set.csv") else "stage3_splits/train_set.csv"
val_path = "val_set.csv" if os.path.exists("val_set.csv") else "stage3_splits/val_set.csv"

train_df_raw = pd.read_csv(train_path)
val_df_raw = pd.read_csv(val_path)

encoders, decoders = {}, {}
for col in cols:
    unique_vals = train_df_raw[col].dropna().astype(str).unique()
    encoders[col] = {val: i for i, val in enumerate(sorted(unique_vals))}
    decoders[col] = {i: val for val, i in encoders[col].items()}

num_classes_dict = {col: len(encoders[col]) for col in cols}

encoder_json_path = os.path.join(LARGE_MODEL_DIR, "label_encoders.json")
with open(encoder_json_path, "w", encoding="utf-8") as f:
    json.dump(encoders, f, ensure_ascii=False, indent=2)

def compute_smoothed_class_weights(df, cols, encoders):
    class_weights_dict = {}
    for col in cols:
        counts = df[col].dropna().astype(str).map(encoders[col]).value_counts()
        num_classes = len(encoders[col])
        total_samples = len(df[col].dropna())

        weights = torch.ones(num_classes, dtype=torch.float)
        for class_id, count in counts.items():
            weights[class_id] = np.sqrt(total_samples / (num_classes * count))

        weights = torch.clamp(weights, min=0.2, max=5.0)
        class_weights_dict[col] = weights
    return class_weights_dict

class_weights_dict = compute_smoothed_class_weights(train_df_raw, cols, encoders)

# ------------------------------------------------------------------------------
# 3. MODEL ARCHITECTURE & MEAN POOLING
# ------------------------------------------------------------------------------
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = (
        attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    )
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return sum_embeddings / sum_mask

class DictaBertMultiTaskUncertainty(nn.Module):
    def __init__(self, model_name, num_classes_dict, class_weights_dict=None):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(self.bert.config.hidden_dropout_prob)

        self.heads = nn.ModuleDict(
            {col: nn.Linear(hidden_size, num_classes_dict[col]) for col in cols}
        )
        self.class_weights_dict = class_weights_dict
        self.log_vars = nn.Parameter(torch.zeros(len(cols)))

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = mean_pooling(outputs, attention_mask)
        pooled_output = self.dropout(pooled_output)

        logits = {col: self.heads[col](pooled_output) for col in cols}

        loss = None
        if labels is not None:
            total_loss = 0.0
            for i, col in enumerate(cols):
                if col not in labels:
                    continue

                valid_mask = labels[col] != -100
                if valid_mask.any():
                    if self.class_weights_dict and col in self.class_weights_dict:
                        weights = self.class_weights_dict[col].to(input_ids.device)
                        loss_fct = nn.CrossEntropyLoss(weight=weights, ignore_index=-100)
                    else:
                        loss_fct = nn.CrossEntropyLoss(ignore_index=-100)

                    raw_head_loss = loss_fct(logits[col], labels[col])
                    precision = torch.exp(-self.log_vars[i])
                    weighted_head_loss = (
                        0.5 * precision * raw_head_loss + 0.5 * self.log_vars[i]
                    )
                    total_loss += weighted_head_loss

            loss = (
                total_loss
                if isinstance(total_loss, torch.Tensor)
                else torch.tensor(0.0, requires_grad=True, device=input_ids.device)
            )

        return {"loss": loss, "logits": logits}

# ------------------------------------------------------------------------------
# 4. DATASET & COLLATOR
# ------------------------------------------------------------------------------
class MultiTaskStage3DatasetClean(Dataset):
    def __init__(self, df, tokenizer, encoders, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.encoders = encoders
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = (
            f"משפט: {str(row['Full_Raw_Sentence'])} | יחידה:"
            f" {str(row['MeaningVal'])} | רפרנט: {str(row['Referent'])}"
        )

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        labels = {}
        for col in cols:
            val = row[col]
            if pd.isna(val):
                labels[col] = torch.tensor(-100, dtype=torch.long)
            else:
                labels[col] = torch.tensor(
                    self.encoders[col].get(str(val), -100), dtype=torch.long
                )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": labels,
        }

def multi_task_collate_fn(batch):
    input_ids = torch.stack([item["input_ids"] for item in batch])
    attention_mask = torch.stack([item["attention_mask"] for item in batch])
    labels = {
        key: torch.stack([item["labels"][key] for item in batch])
        for key in batch[0]["labels"].keys()
    }
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

class MultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs, labels=labels)
        loss = outputs["loss"]
        return (loss, outputs) if return_outputs else loss

# ------------------------------------------------------------------------------
# 5. EXECUTE TRAINING
# ------------------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = DictaBertMultiTaskUncertainty(
    MODEL_NAME, num_classes_dict, class_weights_dict=class_weights_dict
)

train_dataset = MultiTaskStage3DatasetClean(train_df_raw, tokenizer, encoders)
val_dataset = MultiTaskStage3DatasetClean(val_df_raw, tokenizer, encoders)

training_args = TrainingArguments(
    output_dir=TMP_CHECKPOINT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=multi_task_collate_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f"🚀 Starting Training for {MODEL_NAME}...")
trainer.train()

print(f"\n💾 Saving final clean weights and tokenizer to {LARGE_MODEL_DIR}...")
torch.save(model.state_dict(), os.path.join(LARGE_MODEL_DIR, "pytorch_model.bin"))
model.bert.config.save_pretrained(LARGE_MODEL_DIR)
tokenizer.save_pretrained(LARGE_MODEL_DIR)

shutil.rmtree(TMP_CHECKPOINT_DIR, ignore_errors=True)
print(f"🎉 SUCCESS! Model saved in Google Drive: {LARGE_MODEL_DIR}")

# ------------------------------------------------------------------------------
# 6. AUTOMATED EVALUATION & COMPARISON BENCHMARK
# ------------------------------------------------------------------------------
def evaluate_stage3(model_dir, model_name, hub_id):
    if not os.path.exists(model_dir):
        print(f"⚠️ Model directory missing: {model_dir}")
        return None

    print(f"\n⏳ Evaluating Stage 3 ({model_name})...")

    # Load tokenizer & config
    try:
        eval_tok = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
    except Exception:
        eval_tok = AutoTokenizer.from_pretrained(hub_id, trust_remote_code=True)

    with open(os.path.join(model_dir, "label_encoders.json"), "r", encoding="utf-8") as f:
        local_encoders = json.load(f)

    local_num_classes = {col: len(local_encoders[col]) for col in cols}

    eval_mod = DictaBertMultiTaskUncertainty(hub_id, local_num_classes).to(device)
    weights_path = os.path.join(model_dir, "pytorch_model.bin")
    eval_mod.load_state_dict(torch.load(weights_path, map_location=device))
    eval_mod.eval()

    eval_ds = MultiTaskStage3DatasetClean(val_df_raw, eval_tok, local_encoders)
    dataloader = DataLoader(eval_ds, batch_size=8, collate_fn=multi_task_collate_fn)

    dummy = eval_tok(["משפט: בדיקה | יחידה: test | רפרנט: test"], return_tensors="pt").to(device)
    with torch.no_grad():
        _ = eval_mod(dummy["input_ids"], dummy["attention_mask"])

    all_preds = {col: [] for col in cols}
    all_targets = {col: [] for col in cols}

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start_time = time.perf_counter()

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"]

            outputs = eval_mod(input_ids, attention_mask)
            logits = outputs["logits"]

            for col in cols:
                preds = torch.argmax(logits[col], dim=-1).cpu().numpy()
                targs = labels[col].numpy()

                valid_idx = targs != -100
                all_preds[col].extend(preds[valid_idx])
                all_targets[col].extend(targs[valid_idx])

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.perf_counter()

    # Calculate metrics per head
    results = {"Model": model_name}
    acc_scores, f1_scores = [], []

    for col in cols:
        acc = accuracy_score(all_targets[col], all_preds[col])
        macro_f1 = f1_score(all_targets[col], all_preds[col], average="macro", zero_division=0)

        results[f"{col}_Acc"] = f"{acc*100:.1f}%"
        results[f"{col}_f1"] = f"{macro_f1*100:.1f}%"
        acc_scores.append(acc)
        f1_scores.append(macro_f1)

    # Summary metrics
    latency_ms = ((end_time - start_time) / len(eval_ds)) * 1000
    results["Avg Acc ↑"] = f"{np.mean(acc_scores)*100:.2f}%"
    results["Avg Macro F1 ↑"] = f"{np.mean(f1_scores)*100:.2f}%"
    results["Latency ↓"] = f"{latency_ms:.2f} ms"

    del eval_mod, eval_tok
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results

# Run Evaluation
large_res = evaluate_stage3(LARGE_MODEL_DIR, "DictaBERT Large", "dicta-il/dictabert-large")
base_res = evaluate_stage3(BASE_MODEL_DIR, "DictaBERT Base", "dicta-il/dictabert")

eval_data = [res for res in [base_res, large_res] if res is not None]
df_results = pd.DataFrame(eval_data)

print("\n" + "=" * 80)
print("📊 STAGE 3 SUMMARY SLIDE COMPARISON TABLE")
print("=" * 80)
summary_cols = ["Model", "Avg Acc ↑", "Avg Macro F1 ↑", "Latency ↓"]
print(df_results[summary_cols].to_markdown(index=False))

print("\n" + "=" * 80)
print("🎯 DETAILED HEAD-BY-HEAD METRICS (Accuracy / Macro F1)")
print("=" * 80)
detailed_cols = ["Model"] + [c for c in df_results.columns if c not in summary_cols]
print(df_results[detailed_cols].to_markdown(index=False))

💾 Mounting Google Drive...
Mounted at /content/drive


config.json:   0%|          | 0.00/407 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: reconstructing file:   0%|          |  0.00B / 1.50MB            

vocab.txt: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dicta-il/dictabert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



📊 STAGE 3 SUMMARY SLIDE COMPARISON TABLE
| Model           | Avg Acc ↑   | Avg Macro F1 ↑   | Latency ↓   |
|:----------------|:------------|:-----------------|:------------|
| DictaBERT Base  | 87.27%      | 56.20%           | 9.80 ms     |
| DictaBERT Large | 86.91%      | 55.21%           | 33.31 ms    |

🎯 DETAILED HEAD-BY-HEAD METRICS (Accuracy / Macro F1)
| Model           | dim_Acc   | dim_f1   | tr_Acc   | tr_f1   | fr_Acc   | fr_f1   | sr_Acc   | sr_f1   | fe_Acc   | fe_f1   | refLevel_Acc   | refLevel_f1   | ss_Acc   | ss_f1   |
|:----------------|:----------|:---------|:---------|:--------|:---------|:--------|:---------|:--------|:---------|:--------|:---------------|:--------------|:---------|:--------|
| DictaBERT Base  | 78.6%     | 70.4%    | 84.0%    | 51.9%   | 95.8%    | 67.8%   | 92.3%    | 33.5%   | 100.0%   | 50.0%   | 74.8%          | 56.4%         | 85.5%    | 63.3%   |
| DictaBERT Large | 79.0%     | 71.9%    | 83.8%    | 51.6%   | 95.7%    | 69.9%   | 92.4%  

On Test

In [ ]:
# ============================================================
# STAGE 3: TEST SET EVALUATION (BASE VS. LARGE)
# ============================================================
import json
import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from google.colab import drive
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModel, AutoTokenizer

# ------------------------------------------------------------------------------
# 1. SETUP PATHS & DATA
# ------------------------------------------------------------------------------
drive.mount("/content/drive", force_remount=False)

cols = ["dim", "tr", "fr", "sr", "fe", "refLevel", "ss"]

BASE_MODEL_DIR = "/content/drive/MyDrive/Workshop/New_Models/stage3_dictabert_uncertainty"
LARGE_MODEL_DIR = "/content/drive/MyDrive/Workshop/Large/models/stage3_dictabert_large_uncertainty"

test_path = "test_set.csv" if os.path.exists("test_set.csv") else "stage3_splits/test_set.csv"
test_df_raw = pd.read_csv(test_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------------------------
# 2. MODEL ARCHITECTURE & MEAN POOLING
# ------------------------------------------------------------------------------
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = (
        attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    )
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return sum_embeddings / sum_mask

class DictaBertMultiTaskUncertainty(nn.Module):
    def __init__(self, model_name, num_classes_dict, class_weights_dict=None):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(self.bert.config.hidden_dropout_prob)

        self.heads = nn.ModuleDict(
            {col: nn.Linear(hidden_size, num_classes_dict[col]) for col in cols}
        )
        self.class_weights_dict = class_weights_dict
        self.log_vars = nn.Parameter(torch.zeros(len(cols)))

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = mean_pooling(outputs, attention_mask)
        pooled_output = self.dropout(pooled_output)

        logits = {col: self.heads[col](pooled_output) for col in cols}

        loss = None
        if labels is not None:
            total_loss = 0.0
            for i, col in enumerate(cols):
                if col not in labels:
                    continue

                valid_mask = labels[col] != -100
                if valid_mask.any():
                    if self.class_weights_dict and col in self.class_weights_dict:
                        weights = self.class_weights_dict[col].to(input_ids.device)
                        loss_fct = nn.CrossEntropyLoss(weight=weights, ignore_index=-100)
                    else:
                        loss_fct = nn.CrossEntropyLoss(ignore_index=-100)

                    raw_head_loss = loss_fct(logits[col], labels[col])
                    precision = torch.exp(-self.log_vars[i])
                    weighted_head_loss = (
                        0.5 * precision * raw_head_loss + 0.5 * self.log_vars[i]
                    )
                    total_loss += weighted_head_loss

            loss = (
                total_loss
                if isinstance(total_loss, torch.Tensor)
                else torch.tensor(0.0, requires_grad=True, device=input_ids.device)
            )

        return {"loss": loss, "logits": logits}

# ------------------------------------------------------------------------------
# 3. DATASET & COLLATOR
# ------------------------------------------------------------------------------
class MultiTaskStage3DatasetClean(Dataset):
    def __init__(self, df, tokenizer, encoders, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.encoders = encoders
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = (
            f"משפט: {str(row['Full_Raw_Sentence'])} | יחידה:"
            f" {str(row['MeaningVal'])} | רפרנט: {str(row['Referent'])}"
        )

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        labels = {}
        for col in cols:
            val = row[col]
            if pd.isna(val):
                labels[col] = torch.tensor(-100, dtype=torch.long)
            else:
                labels[col] = torch.tensor(
                    self.encoders[col].get(str(val), -100), dtype=torch.long
                )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": labels,
        }

def multi_task_collate_fn(batch):
    input_ids = torch.stack([item["input_ids"] for item in batch])
    attention_mask = torch.stack([item["attention_mask"] for item in batch])
    labels = {
        key: torch.stack([item["labels"][key] for item in batch])
        for key in batch[0]["labels"].keys()
    }
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

# ------------------------------------------------------------------------------
# 4. EVALUATION FUNCTION
# ------------------------------------------------------------------------------
def evaluate_stage3(model_dir, model_name, hub_id, eval_df):
    if not os.path.exists(model_dir):
        print(f"⚠️ Model directory missing: {model_dir}")
        return None

    print(f"\n⏳ Evaluating Stage 3 ({model_name}) on test set...")

    try:
        eval_tok = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
    except Exception:
        eval_tok = AutoTokenizer.from_pretrained(hub_id, trust_remote_code=True)

    encoder_path = os.path.join(model_dir, "label_encoders.json")
    with open(encoder_path, "r", encoding="utf-8") as f:
        local_encoders = json.load(f)

    local_num_classes = {col: len(local_encoders[col]) for col in cols}

    eval_mod = DictaBertMultiTaskUncertainty(hub_id, local_num_classes).to(device)
    weights_path = os.path.join(model_dir, "pytorch_model.bin")
    eval_mod.load_state_dict(torch.load(weights_path, map_location=device))
    eval_mod.eval()

    eval_ds = MultiTaskStage3DatasetClean(eval_df, eval_tok, local_encoders)
    dataloader = DataLoader(eval_ds, batch_size=8, collate_fn=multi_task_collate_fn)

    dummy = eval_tok(["משפט: בדיקה | יחידה: test | רפרנט: test"], return_tensors="pt").to(device)
    with torch.no_grad():
        _ = eval_mod(dummy["input_ids"], dummy["attention_mask"])

    all_preds = {col: [] for col in cols}
    all_targets = {col: [] for col in cols}

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start_time = time.perf_counter()

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"]

            outputs = eval_mod(input_ids, attention_mask)
            logits = outputs["logits"]

            for col in cols:
                preds = torch.argmax(logits[col], dim=-1).cpu().numpy()
                targs = labels[col].numpy()

                valid_idx = targs != -100
                all_preds[col].extend(preds[valid_idx])
                all_targets[col].extend(targs[valid_idx])

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.perf_counter()

    results = {"Model": model_name}
    acc_scores, f1_scores = [], []

    for col in cols:
        acc = accuracy_score(all_targets[col], all_preds[col])
        macro_f1 = f1_score(all_targets[col], all_preds[col], average="macro", zero_division=0)

        results[f"{col}_Acc"] = f"{acc*100:.1f}%"
        results[f"{col}_f1"] = f"{macro_f1*100:.1f}%"
        acc_scores.append(acc)
        f1_scores.append(macro_f1)

    latency_ms = ((end_time - start_time) / len(eval_ds)) * 1000
    results["Avg Acc ↑"] = f"{np.mean(acc_scores)*100:.2f}%"
    results["Avg Macro F1 ↑"] = f"{np.mean(f1_scores)*100:.2f}%"
    results["Latency ↓"] = f"{latency_ms:.2f} ms"

    del eval_mod, eval_tok
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results

# ------------------------------------------------------------------------------
# 5. RUN COMPARISON
# ------------------------------------------------------------------------------
large_res = evaluate_stage3(LARGE_MODEL_DIR, "DictaBERT Large", "dicta-il/dictabert-large", test_df_raw)
base_res = evaluate_stage3(BASE_MODEL_DIR, "DictaBERT Base", "dicta-il/dictabert", test_df_raw)

eval_data = [res for res in [base_res, large_res] if res is not None]
df_results = pd.DataFrame(eval_data)

print("\n" + "=" * 80)
print("📊 STAGE 3 TEST SET SUMMARY SLIDE COMPARISON TABLE")
print("=" * 80)
summary_cols = ["Model", "Avg Acc ↑", "Avg Macro F1 ↑", "Latency ↓"]
print(df_results[summary_cols].to_markdown(index=False))

print("\n" + "=" * 80)
print("🎯 DETAILED HEAD-BY-HEAD METRICS (Accuracy / Macro F1)")
print("=" * 80)
detailed_cols = ["Model"] + [c for c in df_results.columns if c not in summary_cols]
print(df_results[detailed_cols].to_markdown(index=False))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

⏳ Evaluating Stage 3 (DictaBERT Large) on test set...


config.json:   0%|          | 0.00/407 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.74GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dicta-il/dictabert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



📊 STAGE 3 TEST SET SUMMARY SLIDE COMPARISON TABLE
| Model           | Avg Acc ↑   | Avg Macro F1 ↑   | Latency ↓   |
|:----------------|:------------|:-----------------|:------------|
| DictaBERT Base  | 87.43%      | 54.92%           | 8.12 ms     |
| DictaBERT Large | 88.39%      | 64.27%           | 26.19 ms    |

🎯 DETAILED HEAD-BY-HEAD METRICS (Accuracy / Macro F1)
| Model           | dim_Acc   | dim_f1   | tr_Acc   | tr_f1   | fr_Acc   | fr_f1   | sr_Acc   | sr_f1   | fe_Acc   | fe_f1   | refLevel_Acc   | refLevel_f1   | ss_Acc   | ss_f1   |
|:----------------|:----------|:---------|:---------|:--------|:---------|:--------|:---------|:--------|:---------|:--------|:---------------|:--------------|:---------|:--------|
| DictaBERT Base  | 77.4%     | 65.2%    | 84.6%    | 58.2%   | 96.1%    | 67.0%   | 93.5%    | 33.2%   | 99.9%    | 50.0%   | 77.2%          | 59.4%         | 83.3%    | 51.5%   |
| DictaBERT Large | 79.3%     | 68.6%    | 84.6%    | 58.7%   | 96.0%    | 65.4%   